# TensorFlow and Keras: a complete handwritten-digit project

A digit recognizer should learn from examples and accept the same image representation after deployment. Trace the data from raw pixels through batching and the network to class scores, then use validation and held-out errors to assess how well the workflow generalizes.

We will build and evaluate a neural network that recognizes handwritten digits from scikit-learn's bundled 8-by-8 grayscale images. The workflow covers data inspection, leakage-aware train/validation/test splits, batching, model design, early stopping, learning curves, per-class error analysis, model reload, and exportable project artifacts. No dataset downloads, camera, GPU, or account are required; run cells from top to bottom.

**Project input contract:** one 8-by-8 grayscale image per example, represented as `(8, 8, 1)` `float32` values in the range 0 to 16; labels are integers from 0 to 9. The model performs its fixed pixel rescaling internally.

**Success checks:** the trained model should beat the training-majority baseline on held-out data, per-class metrics and a confusion matrix should be inspected, and reloaded model predictions should match the original model. The test set is reserved for final evaluation after model choices are complete. This small benchmark is a learning project, not a production accuracy guarantee.

## 1. Install and import

In a compatible Python environment, run `%pip install tensorflow scikit-learn matplotlib` if needed. TensorFlow wheels depend on Python version and platform; consult the official [installation guide](https://www.tensorflow.org/install/pip) if installation fails. A dedicated virtual environment avoids changing dependencies used by other notebooks.

This notebook uses `tf.keras` consistently. Setting a random seed helps repeatability, but exact numerical results can still vary with hardware and package versions.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)
print("Devices:", [device.device_type for device in tf.config.list_physical_devices()])

## 2. Tensors have a shape and a dtype

A tensor is a multidimensional array. Tensor shapes describe dimensions; dtypes describe the stored values. Most network computations here use float32.

`*` multiplies element by element. `tf.matmul` performs matrix multiplication, combining rows with columns. A Tensor is immutable; a Variable holds values that can be updated.

In [ ]:
a = tf.constant([[1, 2], [3, 4]], dtype=tf.float32)
b = tf.constant([[2, 0], [0, 2]], dtype=tf.float32)
print("Shape:", a.shape, "dtype:", a.dtype)
print("Elementwise product:", (a * b).numpy())
print("Matrix product:", tf.matmul(a, b).numpy())
variable = tf.Variable(3.0)
variable.assign_add(1.0)
print("Updated variable:", variable.numpy())

## 3. Differentiate automatically

Training adjusts weights using gradients of a loss. GradientTape records operations involving watched variables, then computes derivatives.

For $f(w)=w^2+2w$, the derivative is $2w+2$. At $w=3$, it is 8. This small example exposes the mechanism that Keras uses during training.

In [ ]:
w = tf.Variable(3.0)
with tf.GradientTape() as tape:
    value = w**2 + 2*w
gradient = tape.gradient(value, w)
print("Function value:", value.numpy())
print("Gradient:", gradient.numpy())

## 4. Inspect the images before choosing an input shape

Predict the batch shape after adding a channel dimension. Check the pixel range before rescaling. Dividing already-scaled data by the maximum again would satisfy the shape contract but change the meaning of every input.

The bundled dataset contains 8×8 grayscale images and integer labels from 0 to 9. Pixel values range from 0 to 16. These are small benchmark images, not full-resolution photographs.

We add a trailing singleton channel dimension, giving `(examples, height, width, channels)`. The model will accept raw pixel values and perform fixed rescaling internally.

In [ ]:
digits = load_digits()
images = digits.images.astype("float32")[..., np.newaxis]
labels = digits.target.astype("int64")
print("Images:", images.shape, "Labels:", labels.shape)
print("Pixel range:", images.min(), images.max())
print("Classes:", np.unique(labels))

## 5. Split before training

Reserve 20% for the final test. Split the remaining 80% into training and validation, yielding approximately 60/20/20 overall. Stratification retains similar class proportions.

Training data updates weights. Validation data guides stopping and model choices. Test data is reserved for evaluation after those choices. Random splitting is suitable for this demonstration; repeated subjects or time-ordered records may require group-aware or chronological splits.

In [ ]:
indices = np.arange(len(labels))
train_val_idx, test_idx = train_test_split(indices, test_size=0.2, stratify=labels, random_state=42)
train_idx, val_idx = train_test_split(train_val_idx, test_size=0.25,
                                     stratify=labels[train_val_idx], random_state=42)
x_train, y_train = images[train_idx], labels[train_idx]
x_val, y_val = images[val_idx], labels[val_idx]
x_test, y_test = images[test_idx], labels[test_idx]
print("Training:", len(train_idx), "Validation:", len(val_idx), "Test:", len(test_idx))
fig, axes = plt.subplots(2, 5, figsize=(9, 4), layout="constrained")
for ax, image, label in zip(axes.flat, x_train[:10], y_train[:10]):
    ax.imshow(image[:, :, 0], cmap="gray", vmin=0, vmax=16)
    ax.set_title(f"Label {label}")
    ax.axis("off")
plt.show()

## 6. Batch data with tf.data

A Dataset describes how to supply examples. Shuffle the training set, batch it, and prefetch to overlap preparation with model work. Validation and test examples retain their order.

The shuffle buffer covers this small training set. Fixed small thread pools and a one-batch prefetch keep this tutorial's resource use modest. This example still starts with arrays in memory; using Dataset does not make those source arrays disappear.

In [ ]:
def make_dataset(x, y, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        dataset = dataset.shuffle(len(x), seed=42, reshuffle_each_iteration=True)
    options = tf.data.Options()
    options.threading.private_threadpool_size = 1
    options.threading.max_intra_op_parallelism = 1
    return dataset.batch(32).with_options(options).prefetch(1)

train_ds = make_dataset(x_train, y_train, training=True)
val_ds = make_dataset(x_val, y_val)
test_ds = make_dataset(x_test, y_test)
for batch_images, batch_labels in train_ds.take(1):
    print("Image batch:", batch_images.shape, "Label batch:", batch_labels.shape)

## 7. Build a small classifier

Rescaling divides each pixel by the known maximum 16, with no learned statistics. Flatten converts an 8×8×1 image to 64 features. A Dense layer learns weighted combinations; ReLU adds nonlinearity.

The final layer returns ten logits, one per class. Logits are unrestricted scores, not probabilities. This dense model is a compact baseline; a convolutional network can model spatial patterns more explicitly.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(8, 8, 1)),
    tf.keras.layers.Rescaling(1.0 / 16),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10),
], name="digit_classifier")
model.summary()

## 8. Match labels and output scores to the training objective

Identify whether labels are integers or vectors and whether outputs are logits or probabilities. Explain why the configured loss must match both. A decreasing loss cannot rescue an incorrectly defined target representation.

Sparse categorical cross-entropy accepts integer class labels. `from_logits=True` matches the final layer's raw scores. If the model instead returned softmax probabilities, this setting would need to be false.

Adam updates weights using gradients. Accuracy is a reporting metric; the optimizer minimizes the loss, not accuracy directly.

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")])
majority_class = int(np.bincount(y_train, minlength=10).argmax())
validation_baseline = float(np.mean(y_val == majority_class))
print("Training-majority class:", majority_class)
print("Validation majority-class accuracy:", validation_baseline)

## 9. Train with validation and early stopping

An epoch is one pass through the training data. EarlyStopping monitors validation loss, waits five epochs without improvement, and restores weights from the best validation-loss epoch.

Validation examples are evaluated without updating weights. Do not substitute test data here. Repeated model selection can overfit validation data too, so keep track of experiments.

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                                  restore_best_weights=True)
history = model.fit(train_ds, validation_data=val_ds, epochs=40,
                    callbacks=[early_stopping], verbose=0)
best_epoch = int(np.argmin(history.history["val_loss"])) + 1
print("Epochs run:", len(history.history["loss"]))
print("Best validation-loss epoch:", best_epoch)
print("Validation accuracy at that epoch:", history.history["val_accuracy"][best_epoch - 1])

## 10. Compare learning on familiar and unfamiliar examples

Predict the pattern you would expect under overfitting, then inspect both curves. Identify which epoch’s weights are restored. Do not equate the last history entry with the performance of the restored model.

Falling training loss with rising validation loss can indicate overfitting. If both remain poor, consider model capacity, data representation, or optimization. A gap alone does not diagnose the cause.

The histories describe weights at each recorded epoch; the current model uses the restored best-validation weights. They may differ from the final history entry.

In [ ]:
epochs = np.arange(1, len(history.history["loss"]) + 1)
learning_fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for ax, metric, title in zip(axes, ["loss", "accuracy"], ["Cross-entropy loss", "Accuracy"]):
    ax.plot(epochs, history.history[metric], label="Training")
    ax.plot(epochs, history.history[f"val_{metric}"], label="Validation")
    ax.axvline(best_epoch, color="gray", linestyle="--", label="Best validation loss")
    ax.set(xlabel="Epoch", ylabel=title)
    ax.legend()
plt.show()

## 11. Evaluate on the held-out test set

Evaluate once after fixing the workflow. Compare against the training-majority baseline and inspect per-class precision, recall, and F1, not only overall accuracy.

These scores describe this benchmark split. They do not establish performance on different handwriting, image resolutions, or acquisition conditions.

In [ ]:
test_metrics = model.evaluate(test_ds, return_dict=True, verbose=0)
test_logits = model.predict(test_ds, verbose=0)
test_probabilities = tf.nn.softmax(test_logits, axis=-1).numpy()
test_predictions = np.argmax(test_probabilities, axis=1)
test_baseline = float(np.mean(y_test == majority_class))
print("Test metrics:", test_metrics)
print("Test majority-class baseline:", test_baseline)
print(classification_report(y_test, test_predictions, labels=list(range(10)), zero_division=0))
confusion_fig, ax = plt.subplots(figsize=(7, 6), layout="constrained")
ConfusionMatrixDisplay.from_predictions(y_test, test_predictions, labels=list(range(10)),
                                       cmap="Blues", colorbar=False, ax=ax)
ax.set_title("Held-out digit predictions")
plt.show()

## 12. Inspect individual predictions

Softmax produces class probabilities that sum to one, but a high score does not guarantee correctness or calibration. Show predictions alongside true labels to understand mistakes.

If this split has fewer than eight errors, use a few test examples to fill the display. These are diagnostics of the already-computed test results, not a reason to repeatedly retune on that test set.

In [ ]:
error_indices = np.flatnonzero(test_predictions != y_test)
preview_indices = error_indices[:8] if len(error_indices) >= 8 else np.arange(min(8, len(y_test)))
fig, axes = plt.subplots(2, 4, figsize=(10, 5), layout="constrained")
for ax, index in zip(axes.flat, preview_indices):
    prediction = int(test_predictions[index])
    ax.imshow(x_test[index, :, :, 0], cmap="gray", vmin=0, vmax=16)
    ax.set_title(f"True {y_test[index]}, predicted {prediction}\nScore {test_probabilities[index, prediction]:.2f}")
    ax.axis("off")
plt.show()
print("Misclassified test images:", len(error_indices))

## 13. Preserve preprocessing when handing off the model

Before reloading, state the model’s expected image shape and raw pixel range. Compare predictions on fixed inputs before and after reload. A saved network without its input meaning is an incomplete handoff.

The `.keras` format stores model configuration, weights, and training state. The Rescaling layer is inside the model, so inference still expects raw 0–16 pixel values, not separately normalized images.

Save a small input contract alongside the model: image size, channel layout, pixel range, and class mapping matter. The sample passed below is a held-out image, not an external new dataset.

In [ ]:
output_dir = Path.cwd() / "outputs"
output_dir.mkdir(exist_ok=True)
model_path = output_dir / "tensorflow_digits.keras"
model.save(model_path)
reloaded_model = tf.keras.models.load_model(model_path)
sample_logits = reloaded_model(x_test[:5], training=False).numpy()
original_logits = model(x_test[:5], training=False).numpy()
print("Reloaded predictions match:", np.allclose(sample_logits, original_logits, atol=1e-6))
print("Reloaded predicted classes:", np.argmax(sample_logits, axis=1))

## 14. Export a training report

Write learning curves, a confusion matrix, training history, and test predictions alongside the saved model. Source indices identify the rows in the bundled digits dataset.

Running these export cells replaces their named files under `outputs`. All JSON numerical values are converted to ordinary Python types for portability.

In [ ]:
learning_fig.savefig(output_dir / "tensorflow_learning_curves.png", dpi=150)
confusion_fig.savefig(output_dir / "tensorflow_confusion_matrix.png", dpi=150)
report = {"tensorflow_version": tf.__version__, "training_rows": len(train_idx),
          "validation_rows": len(val_idx), "test_rows": len(test_idx), "best_epoch": best_epoch,
          "test_metrics": {key: float(value) for key, value in test_metrics.items()},
          "test_baseline_accuracy": test_baseline,
          "input_contract": {"shape_without_batch": [8, 8, 1], "dtype": "float32",
                             "pixel_range": [0, 16], "class_labels": list(range(10))},
          "history": {key: [float(value) for value in values] for key, values in history.history.items()}}
(output_dir / "tensorflow_training_report.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
np.savez_compressed(output_dir / "tensorflow_test_predictions.npz", source_indices=test_idx,
                    actual=y_test, predicted=test_predictions, probabilities=test_probabilities)
print("Saved model, two PNG figures, JSON report, and NPZ predictions.")

## 15. Practice and troubleshooting

1. Plan a comparison of 32 and 64 hidden units using validation data. Reserve a fresh evaluation set if you begin redesigning the workflow after examining test results.
2. Add Dropout after the hidden layer and compare validation learning curves. Dropout behaves differently during training and inference.
3. Replace Flatten plus Dense with a small Conv2D feature extractor, keeping the same input contract and split.
4. Reload the saved model in another notebook and supply images with the documented shape and scale.

| Symptom | What to check |
| --- | --- |
| Input shape mismatch | Include batch and channel dimensions: `(batch, 8, 8, 1)` |
| Loss behaves unexpectedly | Match integer labels, logits/probabilities, and loss configuration |
| Predictions change after preprocessing | Do not divide by 16 twice; rescaling is inside the model |
| Training succeeds but validation is poor | Inspect overfitting, label consistency, and distribution differences |
| Repeated runs differ slightly | Seeds, hardware, library versions, and nondeterministic operations all matter |
| Package installation has no matching wheel | Check the Python/platform combinations in the official installation guide |
| High softmax score is treated as certainty | Confidence and calibration need separate evaluation |

See the official [TensorFlow beginner guide](https://www.tensorflow.org/tutorials/quickstart/beginner), [automatic differentiation guide](https://www.tensorflow.org/guide/autodiff), [overfitting tutorial](https://www.tensorflow.org/tutorials/keras/overfit_and_underfit), and [model serialization guide](https://www.tensorflow.org/guide/keras/serialization_and_saving).

## Explain what the evidence supports

Compare the model against its baseline and describe a held-out error. Explain why the saved rescaling layer changes what a caller must send at inference time.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: Keep digit recognition useful when scans deteriorate

Use scikit-learn’s bundled handwritten digits, then inject explicitly simulated duplicate deliveries, absent labels, missing pixels and scanner noise. Compare ordinary training with noise augmentation under the same eight-epoch CPU budget.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through the data boundary

The pixel scale is 0–16, so dividing once by 16 produces 0–1 inputs. Dividing again would compress the signal to 0–0.0625: a shape assertion would miss that bug. Split original image IDs before adding duplicates; otherwise the same handwriting can appear in training and evaluation. Remove exact repeated deliveries, quarantine five unlabeled training rows, and estimate missing-pixel medians only from training. A missing label cannot be reconstructed from its row number.

Validation and test noise are generated once and retained. This makes comparisons paired: both models see the same difficult examples. Noise is a controlled stress test, not evidence about a particular real scanner.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
p2_digits=load_digits();p2_ids=np.arange(len(p2_digits.target))
p2_dev,p2_test_ids=train_test_split(p2_ids,test_size=.2,stratify=p2_digits.target,random_state=71)
p2_train_ids,p2_val_ids=train_test_split(p2_dev,test_size=.2,stratify=p2_digits.target[p2_dev],random_state=71)
p2_rng=np.random.default_rng(71)
p2_raw=pd.DataFrame(p2_digits.data[p2_train_ids],columns=[f'pixel_{i}' for i in range(64)])
p2_raw['id']=p2_train_ids;p2_raw['label']=p2_digits.target[p2_train_ids]
p2_raw.loc[:4,'label']=np.nan
p2_raw=pd.concat([p2_raw,p2_raw.iloc[-12:]],ignore_index=True)
p2_clean=p2_raw.drop_duplicates().dropna(subset=['label']).copy()
assert p2_clean.id.is_unique
p2_x=p2_clean.iloc[:,:64].to_numpy(dtype=np.float32)/16
p2_y=p2_clean.label.to_numpy(dtype=np.int64)
p2_missing=p2_rng.random(p2_x.shape)<.04;p2_x[p2_missing]=np.nan
p2_medians=np.nanmedian(p2_x,axis=0)
p2_x=np.where(np.isnan(p2_x),p2_medians,p2_x).astype('float32')
p2_v=p2_digits.data[p2_val_ids].astype('float32')/16
p2_t=p2_digits.data[p2_test_ids].astype('float32')/16
p2_vy=p2_digits.target[p2_val_ids];p2_ty=p2_digits.target[p2_test_ids]
def p2_corrupt(x,rng):
    return np.clip(x+rng.normal(0,.22,x.shape),0,1).astype('float32')
p2_v_bad=p2_corrupt(p2_v,p2_rng);p2_t_bad=p2_corrupt(p2_t,p2_rng)
print({'raw_train_rows':len(p2_raw),'accepted':len(p2_x),'validation':len(p2_v),'test':len(p2_t),'imputed_pixels':int(p2_missing.sum())})
assert np.isfinite(p2_x).all() and p2_x.max()<=1

### Spend the training budget deliberately

The 64→32→10 network has 2,410 parameters: 64×32+32 biases, plus 32×10+10 biases. Both approaches use the same initial seed, architecture, optimizer and epoch limit. The augmented approach draws new bounded noise for each training batch. Select each model's checkpoint by noisy validation cross-entropy, then report both clean and noisy test accuracy once. This explicitly favors robustness during selection; a different operational objective could make a different checkpoint appropriate.

In [ ]:
import tensorflow as tf

def p2_fit(augment):
    tf.keras.utils.set_random_seed(71);rng=np.random.default_rng(71)
    net=tf.keras.Sequential([tf.keras.layers.Input(shape=(64,)),tf.keras.layers.Dense(32,activation='relu'),tf.keras.layers.Dense(10)])
    opt=tf.keras.optimizers.Adam(.01);loss_fn=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    best=float('inf');history=[];started=time.perf_counter()
    for epoch in range(8):
        order=rng.permutation(len(p2_x))
        for start in range(0,len(order),64):
            ix=order[start:start+64];x=p2_corrupt(p2_x[ix],rng) if augment else p2_x[ix]
            with tf.GradientTape() as tape:loss=loss_fn(p2_y[ix],net(x,training=True))
            opt.apply_gradients(zip(tape.gradient(loss,net.trainable_variables),net.trainable_variables))
        val=float(loss_fn(p2_vy,net(p2_v_bad,training=False)));history.append(val)
        if val<best:best=val;state=net.get_weights()
    net.set_weights(state)
    def predict(x):return np.asarray(net(x,training=False)).argmax(1)
    return net,predict,history,time.perf_counter()-started

p2_results=[]
for p2_aug in [False,True]:
    p2_net,p2_predict,p2_history,p2_seconds=p2_fit(p2_aug)
    p2_full=p2_predict(p2_t_bad)
    p2_chunk=np.concatenate([p2_predict(p2_t_bad[i:i+32]) for i in range(0,len(p2_t_bad),32)])
    assert np.array_equal(p2_full,p2_chunk)
    p2_results.append({'augmentation':p2_aug,'best_noisy_validation_loss':min(p2_history),'selected_epoch':int(np.argmin(p2_history))+1,'clean_test_accuracy':accuracy_score(p2_ty,p2_predict(p2_t)),'noisy_test_accuracy':accuracy_score(p2_ty,p2_full),'training_seconds':p2_seconds})
display(pd.DataFrame(p2_results))
print({'full_input_bytes':p2_t_bad.nbytes,'batch_32_input_bytes':p2_t_bad[:32].nbytes,'epochs_per_approach':8})

### Interpret the tradeoff and debug the next failure

Augmentation can improve noisy accuracy while sacrificing clean accuracy; it can also fail under this small budget. Report what the table actually shows. Cross-entropy can improve without changing accuracy because confidence changes before the winning class does. The batch check proves identical class decisions here; its byte counts describe input arrays, not total framework memory or production latency.

**Try:** replace Gaussian noise with an occluded 2×2 patch. **Worked reasoning:** additive-noise training does not guarantee occlusion robustness. Define a fresh validation corruption policy before tuning, retain an untouched final test set, and inspect errors by digit. A useful next competing approach is a small convolutional model, whose locality bias differs from this dense network.